# Notebook 2: Bronze Layer - Ingestion des données brutes

**Objectif**: Ingestérer les données depuis S3 vers la couche Bronze du lakehouse.

## Qu'est-ce que la couche Bronze?

La **couche Bronze** est la première couche du lakehouse:
- Elle stocke les données **brutes**, telles que reçues de la source
- Ajoute des **métadonnées techniques** (date d'ingestion, nom du fichier, etc.)
- Garde le format original (PascalCase, types d'origine)
- Est **partitionnée** par date d'ingestion pour les performances

## Flux de données

```
S3 (CSV) ──► Lecture Spark ──► Enrichissement métadonnées ──► Table Iceberg Bronze
```

---
## 1. Initialisation

In [ ]:
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET

spark = get_spark("bronze-ingestion")

print("✅ Session Spark initialisée")

## 2. Lecture du fichier CSV depuis S3

Le fichier source contient des données de ventes au format CSV.

**Note**: Les options CSV sont importantes pour gérer correctement les guillemets et virgules dans les données.

In [ ]:
# Chemin du fichier source dans S3
source_path = f"s3a://{AWS_BUCKET}/raw/sales/superstore_sales.csv"

# Lecture du CSV avec les options appropriées
sales_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(source_path)
)

# Afficher un échantillon et le schéma
print("=== Données brutes (5 premières lignes) ===")
sales_raw.show(5, truncate=False)

print("\n=== Schéma des données ===")
sales_raw.printSchema()

In [ ]:
# Nombre total d'enregistrements
raw_count = sales_raw.count()
print(f"📊 Nombre d'enregistrements bruts: {raw_count:,}")

## 3. Enrichissement avec métadonnées techniques

La couche Bronze ajoute des colonnes de traçabilité:

| Colonne | Description |
|---------|-------------|
| `ingestion_date` | Date de l'ingestion (partition) |
| `ingestion_ts` | Timestamp exact de l'ingestion |
| `source_file` | Nom du fichier source |
| `source_system` | Système d'origine |
| `batch_id` | Identifiant du batch |

In [ ]:
from pyspark.sql.functions import current_timestamp, current_date, lit

# Enrichissement avec métadonnées
sales_bronze = (
    sales_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("superstore_sales.csv"))
    .withColumn("source_system", lit("superstore_csv"))
    .withColumn("batch_id", lit("batch_001"))
)

print("=== Données enrichies (avec métadonnées) ===")
sales_bronze.select("Order ID", "Order Date", "Sales", "ingestion_date", "ingestion_ts", "batch_id")\
    .show(5, truncate=False)

## 4. Suppression de l'ancienne table (si elle existe)

Pour ce notebook de démonstration, on repart de zéro.

In [ ]:
# Supprimer la table Bronze si elle existe
spark.sql("DROP TABLE IF EXISTS nessie.bronze.sales")

print("✅ Ancienne table Bronze supprimée (si elle existait)")

## 5. Création de la table Iceberg Bronze

**Points clés**:
- Format: **Iceberg** (format de table ACID)
- Partition: par `ingestion_date` (optimise les requêtes par date)
- Catalogue: **Nessie** (versionnement automatique)

In [ ]:
from pyspark.sql.functions import col

# Créer la table Iceberg partitionnée par ingestion_date
(
    sales_bronze.writeTo("nessie.bronze.sales")
    .using("iceberg")
    .partitionedBy(col("ingestion_date"))
    .create()
)

print("✅ Table Bronze créée: nessie.bronze.sales")

## 6. Vérification de la table Bronze

In [ ]:
# Compter les enregistrements
bronze_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.bronze.sales").first()[0]
print(f"📊 Nombre d'enregistrements dans Bronze: {bronze_count:,}")

In [ ]:
# Échantillon de données
print("=== Échantillon de la table Bronze ===")
spark.sql("SELECT * FROM nessie.bronze.sales LIMIT 5").show(truncate=False)

## 7. Inspecter les métadonnées Iceberg

Iceberg conserve un historique de toutes les modifications. Chaque écriture crée un nouveau **snapshot**.

In [ ]:
# Historique des snapshots
print("=== Historique des snapshots (Time Travel!) ===")
spark.sql("SELECT * FROM nessie.bronze.sales.history").show(truncate=False)

print("\n=== Détails du snapshot ===")
spark.sql("SELECT committed_at, snapshot_id, operation, summary FROM nessie.bronze.sales.snapshots").show(truncate=False)

## 8. Simulation d'une nouvelle ingestion

Ajoutons quelques enregistrements pour créer un deuxième snapshot. Cela démontre la capacité d'**append** d'Iceberg.

In [ ]:
# Ajouter 5 enregistrements (simulation d'une nouvelle ingestion)
sales_bronze.limit(5).writeTo("nessie.bronze.sales").append()

print("✅ Nouvelle ingestion effectuée (append)")

In [ ]:
# Vérifier le nouvel historique
print("=== Historique mis à jour ===")
spark.sql("SELECT * FROM nessie.bronze.sales.history").show(truncate=False)

new_count = spark.sql("SELECT COUNT(*) as cnt FROM nessie.bronze.sales").first()[0]
print(f"\n📊 Nouveau total d'enregistrements: {new_count:,} (+{new_count - bronze_count})")

## 9. Résumé

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         COUCHE BRONZE - INGESTION TERMINÉE                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  Table               nessie.bronze.sales                   ║
║  Format              Iceberg (ACID, Time Travel)           ║
║  Partition           ingestion_date                        ║
║  Enregistrements     {bronze_count:,} + 5 (démo)              ║
║  Snapshots           2 (capacité de time travel!)           ║
║                                                            ║
║  Métadonnées ajoutées:                                     ║
║    • ingestion_date, ingestion_ts                         ║
║    • source_file, source_system, batch_id                 ║
║                                                            ║
╚══════════════════════════════════════════════════════════════╝
""")

print("\n➡️ Prochaine étape: Exécuter '03_silver_transformations.ipynb'")